# Tutorial on PyTorch Geometric


## Data Handling of Graphs


A graph is used to model pairwise relations (edges) between objects (nodes). A single graph in PyG is described by an instance of torch_geometric.data.Data, which holds the following attributes by default:

- data.x: Node feature matrix with shape [num_nodes, num_node_features]
- data.edge_index: Graph connectivity in COO format with shape [2, num_edges] and type torch.long
- data.edge_attr: Edge feature matrix with shape [num_edges, num_edge_features]
- data.y: Target to train against (may have arbitrary shape), e.g., node-level targets of shape [num_nodes, *] or graph-level targets of shape [1, *]
- data.pos: Node position matrix with shape [num_nodes, num_dimensions]

None of these attributes are required. In fact, the Data object is not even restricted to these attributes. We can, e.g., extend it by data.face to save the connectivity of triangles from a 3D mesh in a tensor with shape `[3, num_faces]` and type `torch.long`.

Note

- PyTorch and torchvision define an example as a tuple of an image and a target. We omit this notation in PyG to allow for various data structures in a clean and understandable way.

We show a simple example of an unweighted and undirected graph with three nodes and four edges. Each node contains exactly one feature:


![image](https://pytorch-geometric.readthedocs.io/en/latest/_images/graph.svg)


In [ ]:
import torch
from torch_geometric.data import Data

edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[-1], [0], [1]], dtype=torch.float)

data = Data(x=x, edge_index=edge_index)
print(data)

Data(x=[3, 1], edge_index=[2, 4])


Note that edge_index, i.e. the tensor defining the source and target nodes of all edges, is not a list of index tuples. If you want to write your indices this way, you should transpose and call contiguous on it before passing them to the data constructor:


In [ ]:
edge_index = torch.tensor([[0, 1], [1, 0], [1, 2], [2, 1]], dtype=torch.long)
x = torch.tensor([[-1], [0], [1]], dtype=torch.float)

data = Data(x=x, edge_index=edge_index.t().contiguous())
print(data)

Data(x=[3, 1], edge_index=[2, 4])


Although the graph has only two edges, we need to define four index tuples to account for both directions of a edge.

- You can print out your data object anytime and receive a short information about its attributes and their shapes.

Note that it is necessary that the elements in edge_index only hold indices in the range { 0, ..., num_nodes - 1}. This is needed as we want our final data representation to be as compact as possible, e.g., we want to index the source and destination node features of the first edge (0, 1) via x[0] and x[1], respectively. You can always check that your final Data objects fulfill these requirements by running validate():


In [ ]:
data.validate(raise_on_error=True)

True

Besides holding a number of node-level, edge-level or graph-level attributes, Data provides a number of useful utility functions, e.g.:


In [ ]:
print(data.keys())

['edge_index', 'x']


In [ ]:
print(data["x"])

tensor([[-1.],
        [ 0.],
        [ 1.]])


In [ ]:
for key, item in data:
    print(f"{key} found in data")

x found in data
edge_index found in data


In [ ]:
"edge_attr" in data

False

In [ ]:
data.num_nodes

3

In [ ]:
data.num_node_features

1

In [ ]:
data.has_isolated_nodes()

False

In [ ]:
data.has_self_loops()

False

In [ ]:
data.is_directed()

False

In [ ]:
# Transfer data object to GPU.
device = torch.device("cuda")
data = data.to(device)

You can find a complete list of all methods at torch_geometric.data.Data.


## Common Benchmark Datasets


PyG contains a large number of common benchmark datasets, e.g., all Planetoid datasets (Cora, Citeseer, Pubmed), all graph classification datasets from TUDatasets and their cleaned versions, the QM7 and QM9 dataset, and a handful of 3D mesh/point cloud datasets like FAUST, ModelNet10/40 and ShapeNet.

Initializing a dataset is straightforward. An initialization of a dataset will automatically download its raw files and process them to the previously described `Data` format. E.g., to load the ENZYMES dataset (consisting of 600 graphs within 6 classes), type:


In [ ]:
from torch_geometric.datasets import TUDataset

dataset = TUDataset(root="/tmp/ENZYMES", name="ENZYMES")
print(dataset, len(dataset))

ENZYMES(600) 600


In [ ]:
print(dataset.num_classes, dataset.num_node_features)

6 3


We now have access to all 600 graphs in the dataset:


In [ ]:
data = dataset[0]
print(data)
print(data.is_undirected())

Data(edge_index=[2, 168], x=[37, 3], y=[1])
True


We can see that the first graph in the dataset contains 37 nodes, each one having 3 features. There are 168/2 = 84 undirected edges and the graph is assigned to exactly one class. In addition, the data object is holding exactly one graph-level target.

We can even use slices, long or bool tensors to split the dataset. E.g., to create a 90/10 train/test split, type:


In [ ]:
train_dataset = dataset[:540]
test_dataset = dataset[540:]
print(train_dataset, test_dataset)

ENZYMES(540) ENZYMES(60)


If you are unsure whether the dataset is already shuffled before you split, you can randomly permute it by running:


In [ ]:
dataset = dataset.shuffle()
print(dataset)

ENZYMES(600)


This is equivalent of doing:


In [ ]:
perm = torch.randperm(len(dataset))
dataset = dataset[perm]
print(dataset)

ENZYMES(600)


Let’s try another one! Let’s download Cora, the standard benchmark dataset for semi-supervised graph node classification:


In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="/tmp/Cora", name="Cora")
print(dataset, len(dataset))
print(dataset.num_classes, dataset.num_node_features)

Cora() 1
7 1433


Here, the dataset contains only a single, undirected citation graph:


In [ ]:
data = dataset[0]
print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [ ]:
data.is_undirected()

True

In [ ]:
print(
    "train : validation : test = {} : {} : {}".format(
        data.train_mask.sum().item(),
        data.val_mask.sum().item(),
        data.test_mask.sum().item(),
    )
)

train : validation : test = 140 : 500 : 1000


This time, the Data objects holds a label for each node, and additional node-level attributes: `train_mask`, `val_mask` and `test_mask`, where

- `train_mask` denotes against which nodes to train (140 nodes),
- `val_mask` denotes which nodes to use for validation, e.g., to perform early stopping (500 nodes),
- `test_mask` denotes against which nodes to test (1000 nodes).


## Mini-batches


Neural networks are usually trained in a batch-wise fashion. PyG achieves parallelization over a mini-batch by creating sparse block diagonal adjacency matrices (defined by `edge_index`) and concatenating feature and target matrices in the node dimension. This composition allows differing number of nodes and edges over examples in one batch:


$$\begin{split}\mathbf{A} = \begin{bmatrix} \mathbf{A}_1 & & \\ & \ddots & \\ & & \mathbf{A}_n \end{bmatrix}, \qquad \mathbf{X} = \begin{bmatrix} \mathbf{X}_1 \\ \vdots \\ \mathbf{X}_n \end{bmatrix}, \qquad \mathbf{Y} = \begin{bmatrix} \mathbf{Y}_1 \\ \vdots \\ \mathbf{Y}_n \end{bmatrix}\end{split}$$


PyG contains its own `torch_geometric.loader.DataLoader`, which already takes care of this concatenation process. Let’s learn about it in an example:


In [ ]:
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader

dataset = TUDataset(root="/tmp/ENZYMES", name="ENZYMES", use_node_attr=True)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for batch in loader:
    print(batch, batch.num_graphs)

DataBatch(edge_index=[2, 4000], x=[1038, 21], y=[32], batch=[1038], ptr=[33]) 32
DataBatch(edge_index=[2, 4462], x=[1141, 21], y=[32], batch=[1141], ptr=[33]) 32
DataBatch(edge_index=[2, 3854], x=[1010, 21], y=[32], batch=[1010], ptr=[33]) 32
DataBatch(edge_index=[2, 4100], x=[1097, 21], y=[32], batch=[1097], ptr=[33]) 32
DataBatch(edge_index=[2, 3560], x=[936, 21], y=[32], batch=[936], ptr=[33]) 32
DataBatch(edge_index=[2, 3870], x=[1045, 21], y=[32], batch=[1045], ptr=[33]) 32
DataBatch(edge_index=[2, 3912], x=[1015, 21], y=[32], batch=[1015], ptr=[33]) 32
DataBatch(edge_index=[2, 3550], x=[1018, 21], y=[32], batch=[1018], ptr=[33]) 32
DataBatch(edge_index=[2, 4034], x=[1084, 21], y=[32], batch=[1084], ptr=[33]) 32
DataBatch(edge_index=[2, 4060], x=[1113, 21], y=[32], batch=[1113], ptr=[33]) 32
DataBatch(edge_index=[2, 4154], x=[1034, 21], y=[32], batch=[1034], ptr=[33]) 32
DataBatch(edge_index=[2, 4080], x=[1067, 21], y=[32], batch=[1067], ptr=[33]) 32
DataBatch(edge_index=[2, 3650]

`torch_geometric.data.Batch` inherits from `torch_geometric.data.Data` and contains an additional attribute called `batch`.

`batch` is a column vector which maps each node to its respective graph in the batch:

$$
\mathrm{batch} = {\begin{bmatrix} 0 & \cdots & 0 & 1 & \cdots & n - 2 & n -1 & \cdots & n - 1 \end{bmatrix}}^{\top}
$$

You can use it to, e.g., average node features in the node dimension for each graph individually:


In [ ]:
from torch_geometric.utils import scatter

dataset = TUDataset(root="/tmp/ENZYMES", name="ENZYMES", use_node_attr=True)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for data in loader:
    print(data, data.num_graphs)
    x = scatter(data.x, data.batch, dim=0, reduce="mean")
    print(x.size())

DataBatch(edge_index=[2, 3864], x=[1028, 21], y=[32], batch=[1028], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 3972], x=[1021, 21], y=[32], batch=[1021], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 4446], x=[1211, 21], y=[32], batch=[1211], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 3834], x=[958, 21], y=[32], batch=[958], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 4046], x=[1063, 21], y=[32], batch=[1063], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 4110], x=[1082, 21], y=[32], batch=[1082], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 3668], x=[933, 21], y=[32], batch=[933], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 3368], x=[893, 21], y=[32], batch=[893], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 4136], x=[1052, 21], y=[32], batch=[1052], ptr=[33]) 32
torch.Size([32, 21])
DataBatch(edge_index=[2, 4398], x=[1125, 21], y=[32], batch=[1125], ptr=[33]) 32
torch.S

You can learn more about the internal batching procedure of PyG, e.g., how to modify its behavior, here. For documentation of scatter operations, we refer the interested reader to the torch_scatter documentation.


## Data Transforms


Transforms are a common way in torchvision to transform images and perform augmentation. PyG comes with its own transforms, which expect a Data object as input and return a new transformed `Data` object. Transforms can be chained together using `torch_geometric.transforms.Compose` and are applied before saving a processed dataset on disk (`pre_transform`) or before accessing a graph in a dataset (`transform`).

Let’s look at an example, where we apply transforms on the ShapeNet dataset (containing 17,000 3D shape point clouds and per point labels from 16 shape categories).


In [ ]:
from torch_geometric.datasets import ShapeNet

dataset = ShapeNet(root="/tmp/ShapeNet", categories=["Airplane"])
dataset[0]
# >>> Data(pos=[2518, 3], y=[2518])

We can convert the point cloud dataset into a graph dataset by generating nearest neighbor graphs from the point clouds via transforms:


In [ ]:
import torch_geometric.transforms as T

dataset = ShapeNet(root="/tmp/ShapeNet", categories=["Airplane"], pre_transform=T.KNNGraph(k=6))
dataset[0]
# >>> Data(edge_index=[2, 15108], pos=[2518, 3], y=[2518])

Note

- We use the pre_transform to convert the data before saving it to disk (leading to faster loading times). Note that the next time the dataset is initialized it will already contain graph edges, even if you do not pass any transform. If the pre_transform does not match with the one from the already processed dataset, you will be given a warning.

In addition, we can use the transform argument to randomly augment a Data object, e.g., translating each node position by a small number:


In [ ]:
dataset = ShapeNet(
    root="/tmp/ShapeNet", categories=["Airplane"], pre_transform=T.KNNGraph(k=6), transform=T.RandomJitter(0.01)
)

dataset[0]
# >>> Data(edge_index=[2, 15108], pos=[2518, 3], y=[2518])

You can find a complete list of all implemented transforms at `torch_geometric.transforms`.


## Learning Methods on Graphs


After learning about data handling, datasets, loader and transforms in PyG, it’s time to implement our first graph neural network!

We will use a simple GCN layer and replicate the experiments on the Cora citation dataset. For a high-level explanation on GCN, have a look at its [blog post](http://tkipf.github.io/graph-convolutional-networks/).

We first need to load the Cora dataset:


In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="/tmp/Cora", name="Cora")
print(dataset)

Note that we do not need to use transforms or a dataloader. Now let’s implement a two-layer GCN:


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

The constructor defines two `GCNConv` layers which get called in the forward pass of our network. Note that the non-linearity is not integrated in the `conv` calls and hence needs to be applied afterwards (something which is consistent across all operators in PyG).

Here, we chose to use ReLU as our intermediate non-linearity and finally output a softmax distribution over the number of classes. Let’s train this model on the training nodes for 200 epochs:


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCN().to(device)
data = dataset[0].to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

Finally, we can evaluate our model on the test nodes:


In [ ]:
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f"Accuracy: {acc:.4f}")

This is all it takes to implement your first graph neural network. The easiest way to learn more about Graph Neural Networks is to study the examples in the `examples/` directory and to browse `torch_geometric.nn`. Happy hacking!
